In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

DATA_DIR = Path("../data")
FIGURE_DIR = Path("../outputs/figures")
TABLE_DIR = Path("../outputs/tables")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_parquet(
    DATA_DIR / "customer_retention_dataset.parquet"
)

print(customers.shape)
display(customers.head())

(5256, 18)


,customer_id,first_order_id,first_order_date,first_order_value,first_order_quantity,first_order_unique_products,first_order_product_lines,first_order_average_item_price,first_order_country,first_order_weekday,first_order_month,first_order_hour,first_order_is_weekend,second_order_date,days_to_second_order,available_followup_days,has_full_90d_followup,repeat_within_90d
0,12346,499763,2010-03-02 13:08:00,27.05,5,5,5,5.410000,United Kingdom,Tuesday,3,13,False,2010-06-28 13:53:00,118.031250,646.987500,True,0
1,12347,529924,2010-10-31 14:20:00,611.53,509,40,40,1.201434,Iceland,Sunday,10,14,True,2010-12-07 14:57:00,37.025694,403.937500,True,1
2,12348,524140,2010-09-27 14:59:00,221.16,372,19,19,0.594516,Finland,Monday,9,14,False,2010-12-16 19:09:00,80.173611,437.910417,True,1
3,12349,506394,2010-04-29 13:20:00,1068.52,473,46,46,2.259027,Italy,Thursday,4,13,False,2010-10-28 08:23:00,181.793750,588.979167,True,0
4,12350,543037,2011-02-02 16:01:00,294.40,196,16,16,1.502041,Norway,Wednesday,2,16,False,NaT,NaN,309.867361,True,0


In [2]:
assert customers["customer_id"].is_unique
assert customers["repeat_within_90d"].isin([0, 1]).all()
assert customers["available_followup_days"].ge(90).all()

In [3]:
numeric_features = [
    "first_order_value",
    "first_order_quantity",
    "first_order_unique_products",
    "first_order_product_lines",
    "first_order_average_item_price",
]

categorical_features = [
    "first_order_country",
    "first_order_weekday",
    "first_order_month",
    "first_order_hour",
]

feature_columns = (
    numeric_features
    + categorical_features
)

target_column = "repeat_within_90d"

In [4]:
train_end = pd.Timestamp("2010-12-01")
validation_end = pd.Timestamp("2011-05-01")

train_mask = (
    customers["first_order_date"] < train_end
)

validation_mask = (
    (customers["first_order_date"] >= train_end)
    & (customers["first_order_date"] < validation_end)
)

test_mask = (
    customers["first_order_date"] >= validation_end
)

In [5]:
X_train = customers.loc[
    train_mask, feature_columns
].copy()

y_train = customers.loc[
    train_mask, target_column
].copy()

X_validation = customers.loc[
    validation_mask, feature_columns
].copy()

y_validation = customers.loc[
    validation_mask, target_column
].copy()

X_test = customers.loc[
    test_mask, feature_columns
].copy()

y_test = customers.loc[
    test_mask, target_column
].copy()

In [6]:
assert (
    train_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
).eq(1).all()

assert len(X_train) + len(X_validation) + len(X_test) == len(customers)

assert customers.loc[
    train_mask, "first_order_date"
].max() < customers.loc[
    validation_mask, "first_order_date"
].min()

assert customers.loc[
    validation_mask, "first_order_date"
].max() < customers.loc[
    test_mask, "first_order_date"
].min()

In [7]:
split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "customers": [
        len(X_train),
        len(X_validation),
        len(X_test),
    ],
    "start_date": [
        customers.loc[train_mask, "first_order_date"].min(),
        customers.loc[validation_mask, "first_order_date"].min(),
        customers.loc[test_mask, "first_order_date"].min(),
    ],
    "end_date": [
        customers.loc[train_mask, "first_order_date"].max(),
        customers.loc[validation_mask, "first_order_date"].max(),
        customers.loc[test_mask, "first_order_date"].max(),
    ],
    "repeat_rate": [
        y_train.mean(),
        y_validation.mean(),
        y_test.mean(),
    ],
})

split_summary

,split,customers,start_date,end_date,repeat_rate
0,Train,4239,2009-12-01 07:45:00,2010-11-30 18:46:00,0.475348
1,Validation,558,2010-12-01 12:36:00,2011-04-28 16:40:00,0.353047
2,Test,459,2011-05-01 13:45:00,2011-09-09 14:14:00,0.507625


## Temporal validation strategy

Customers were divided chronologically according to their first-order date.
Earlier cohorts form the training set, subsequent cohorts form the validation
set, and the latest eligible cohorts form the final test set.

A random split would mix customers acquired at different points in time and
could overstate performance if retention patterns, seasonality, or customer
composition change. The test set therefore approximates deploying a model on
future customer cohorts.

The test set will remain untouched until all preprocessing and model choices
have been finalised using the training and validation sets.

In [8]:
training_repeat_rate = y_train.mean()

baseline_probabilities = {
    "Train": np.full(len(y_train), training_repeat_rate),
    "Validation": np.full(
        len(y_validation),
        training_repeat_rate
    ),
    "Test": np.full(
        len(y_test),
        training_repeat_rate
    ),
}

In [9]:
def evaluate_probabilities(y_true, probabilities, threshold=0.5):
    predictions = (
        np.asarray(probabilities) >= threshold
    ).astype(int)

    return {
        "customers": len(y_true),
        "positive_rate": y_true.mean(),
        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),
        "pr_auc": average_precision_score(
            y_true,
            probabilities
        ),
        "log_loss": log_loss(
            y_true,
            probabilities
        ),
        "brier_score": brier_score_loss(
            y_true,
            probabilities
        ),
        "accuracy_at_0.5": accuracy_score(
            y_true,
            predictions
        ),
    }

In [10]:
baseline_results = pd.DataFrame([
    {
        "split": "Train",
        **evaluate_probabilities(
            y_train,
            baseline_probabilities["Train"]
        ),
    },
    {
        "split": "Validation",
        **evaluate_probabilities(
            y_validation,
            baseline_probabilities["Validation"]
        ),
    },
    {
        "split": "Test",
        **evaluate_probabilities(
            y_test,
            baseline_probabilities["Test"]
        ),
    },
])

baseline_results

,split,customers,positive_rate,roc_auc,pr_auc,log_loss,brier_score,accuracy_at_0.5
0,Train,4239,0.475348,0.5,0.475348,0.691931,0.249392,0.524652
1,Validation,558,0.353047,0.5,0.353047,0.679862,0.243362,0.646953
2,Test,459,0.507625,0.5,0.507625,0.695117,0.250984,0.492375


In [11]:
split_summary.to_csv(
    TABLE_DIR / "temporal_split_summary.csv",
    index=False
)

baseline_results.to_csv(
    TABLE_DIR / "baseline_results.csv",
    index=False
)

## Baseline interpretation

The repeat rate changed substantially across time: 47.53% in training,
35.30% in validation, and 50.76% in testing. This indicates temporal
variation in either customer composition or repeat-purchasing behaviour.

The constant-probability baseline had no ranking ability, with ROC-AUC
equal to 0.5. Its validation accuracy of 64.70% arose because the training
repeat rate was below the 0.5 classification threshold, causing every
customer to be classified as a non-repeat customer. Accuracy is therefore
not an appropriate primary metric for this problem.

The test set will not be used again until model and preprocessing choices
have been finalised using the training and validation sets.